# TinyRPSNet

In [187]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler

In [188]:
height = 64
width = 64

train_transform = transforms.Compose([
	transforms.Grayscale(num_output_channels=1),
	transforms.RandomRotation(degrees=20, fill=0),  # data augmentation
	transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3
    ),
    transforms.Resize((height, width)),
    transforms.ToTensor(),
	transforms.Normalize(mean=[0.5], std=[0.5])
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((height, width)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


mandens_train_dataset = datasets.ImageFolder("data/mandens/train", transform=train_transform)
home_cooked_train_dataset = datasets.ImageFolder("data/home_cooked/train", transform=train_transform)
mandens_test_dataset = datasets.ImageFolder("data/mandens/test", transform=test_transform)
home_cooked_test_dataset = datasets.ImageFolder("data/home_cooked/test", transform=test_transform)

combined_train_dataset = ConcatDataset([mandens_train_dataset, mandens_test_dataset, home_cooked_train_dataset])

weights = [0.15] * len(mandens_train_dataset) + \
          [0.15] * len(mandens_test_dataset) + \
          [0.7] * len(home_cooked_train_dataset) # more representative

sampler = WeightedRandomSampler(weights, num_samples=len(combined_train_dataset), replacement=True)

batch_size = 16

train_loader = DataLoader(
    combined_train_dataset,
    batch_size=batch_size,
	sampler=sampler,
    num_workers=4
)
test_loader = DataLoader(
    home_cooked_test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

In [189]:
print(f"Number of instances: \n Training: {len(train_loader.dataset)}, Testing: {len(test_loader.dataset)}\n")
print(f"Number of batches: \n Training: {len(train_loader)}, Testing: {len(test_loader)}")

Number of instances: 
 Training: 3005, Testing: 75

Number of batches: 
 Training: 188, Testing: 5


In [190]:
class TinyRPSNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # Initial conv
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=3,
            stride=2,
            padding=1
        )
        self.bn1 = nn.BatchNorm2d(8)

        # Depthwise + Pointwise block 1
        self.dw1 = nn.Conv2d(
            in_channels=8,
            out_channels=8,
            kernel_size=3,
            padding=1,
            groups=8  # depthwise
        )
        self.bn_dw1 = nn.BatchNorm2d(8)
        self.pw1 = nn.Conv2d(
            in_channels=8,
            out_channels=16,
            kernel_size=1
        )
        self.bn_pw1 = nn.BatchNorm2d(16)
        # Depthwise + Pointwise block 2
        self.dw2 = nn.Conv2d(
            in_channels=16,
            out_channels=16,
            kernel_size=3,
            padding=1,
            groups=16
        )
        self.bn_dw2 = nn.BatchNorm2d(16)
        self.pw2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=1
        )
        self.bn_pw2 = nn.BatchNorm2d(32)

        # Global average pooling + classifier
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        # x: (B, 1, 64, 64)
        self.act = nn.LeakyReLU(0.1)

        x = self.act(self.bn1(self.conv1(x)))
        x = self.act(self.bn_pw1(self.pw1(self.bn_dw1(self.dw1(x)))))
        x = self.act(self.bn_pw2(self.pw2(self.bn_dw2(self.dw2(x)))))

        x = self.pool(x)           # (B, 32, 1, 1)
        x = torch.flatten(x, 1)    # (B, 32)
        x = self.fc(x)             # (B, num_classes)

        return x  # logits (no softmax here)

Sanity check

In [191]:
def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

model = TinyRPSNet(num_classes=3)
dummy_input = torch.randn(1, 1, 64, 64)
logits = model(dummy_input)
logits.shape

torch.Size([1, 3])

# Instantiating the model

In [192]:
device = "cpu"
print(f"Using {device} device")
model = TinyRPSNet(num_classes=3).to(device)
model.apply(init_weights)

print(model)

Using cpu device
TinyRPSNet(
  (conv1): Conv2d(1, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dw1): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=8)
  (bn_dw1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pw1): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
  (bn_pw1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dw2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16)
  (bn_dw2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pw2): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1))
  (bn_pw2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): AdaptiveAvgPool2d(output_size=1)
  (fc): Linear(in_features=32, out_features=3, bias=True)
)


In [193]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [194]:
def train(data_loader: DataLoader):
	model.train()
	size = len(data_loader.dataset)
	for batch, (X, y) in enumerate(data_loader):
		X, y = X.to(device), y.to(device)

		#Prediction error
		pred = model(X)
		loss = loss_fn(pred, y)

		#Backpropagation
		loss.backward()
		optimizer.step()
		optimizer.zero_grad()

		if batch % 50 == 0:
			loss, current = loss.item(), (batch + 1) * len(X)
			print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [195]:
def test(data_loader: DataLoader):
	model.eval()
	size = len(data_loader.dataset)
	num_batches = len(data_loader)
	test_loss, correct = 0, 0
	
	with torch.no_grad():
		for X, y in data_loader:
			X, y = X.to(device), y.to(device)
			pred = model(X)
			loss = loss_fn(pred, y)
			test_loss += loss.item()
			correct += (pred.argmax(1) == y).type(torch.float).sum().item()
			
	test_loss /= num_batches
	correct /= size
	print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
	return correct


In [196]:
epochs = 1000
best_acc = 0.84
best_epoch = 0
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader)
    correct = test(test_loader)
    if correct > best_acc:
        print("New best accuracy! Saving model...")
        best_acc = correct
        best_epoch = t + 1
        torch.save(model.state_dict(), "model.pth")
print(f"Best Accuracy: {(100*best_acc):>0.1f}% at epoch {best_epoch}")
print("Done!")

Epoch 1
-------------------------------
loss: 1.302007 [   16/ 3005]
loss: 1.138997 [  816/ 3005]
loss: 1.097689 [ 1616/ 3005]
loss: 1.106726 [ 2416/ 3005]
Test Error: 
 Accuracy: 33.3%, Avg loss: 1.097151 

Epoch 2
-------------------------------
loss: 1.050793 [   16/ 3005]
loss: 1.108162 [  816/ 3005]
loss: 1.110353 [ 1616/ 3005]
loss: 1.077200 [ 2416/ 3005]
Test Error: 
 Accuracy: 33.3%, Avg loss: 1.103514 

Epoch 3
-------------------------------
loss: 1.064514 [   16/ 3005]
loss: 1.075464 [  816/ 3005]
loss: 1.068849 [ 1616/ 3005]
loss: 1.094049 [ 2416/ 3005]
Test Error: 
 Accuracy: 34.7%, Avg loss: 1.092283 

Epoch 4
-------------------------------
loss: 1.085045 [   16/ 3005]
loss: 1.075245 [  816/ 3005]
loss: 1.065933 [ 1616/ 3005]
loss: 1.068794 [ 2416/ 3005]
Test Error: 
 Accuracy: 30.7%, Avg loss: 1.097813 

Epoch 5
-------------------------------
loss: 1.052104 [   16/ 3005]
loss: 1.111784 [  816/ 3005]
loss: 1.074663 [ 1616/ 3005]
loss: 1.121913 [ 2416/ 3005]
Test Error: 

KeyboardInterrupt: 

In [197]:
best_acc, best_epoch

(0.9333333333333333, 939)

In [ ]:
# Approx memory footprint in RAM (MB) for parameters + buffers
total_params = sum(p.numel() for p in model.parameters())
param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
size_mb = (param_bytes + buffer_bytes) / (1024 ** 2)

print("params:", total_params)
print(f"Model size (params+buffers): {size_mb:.3f} MB")

params: 1267
Model size (params+buffers): 0.005 MB


Reloading & evaluating model

In [198]:
# Evaluate the best model on the test set and collect predictions and labels for confusion matrix
model = TinyRPSNet(num_classes=3).to(device)
model.load_state_dict(torch.load("model.pth"))
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [199]:
from sklearn.metrics import confusion_matrix, classification_report

# Generate confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

# Print classification report
class_names = home_cooked_test_dataset.classes  # ['paper', 'rock', 'scissors']
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

Confusion Matrix:
[[22  1  2]
 [ 1 23  1]
 [ 0  0 25]]

Classification Report:
              precision    recall  f1-score   support

       paper       0.96      0.88      0.92        25
        rock       0.96      0.92      0.94        25
    scissors       0.89      1.00      0.94        25

    accuracy                           0.93        75
   macro avg       0.94      0.93      0.93        75
weighted avg       0.94      0.93      0.93        75

